In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# DN = 'C://work/dev/python/progs/texts/sec_bert/'
DN = '/home/jovyan/work/sec_bert/'

import os
os.chdir(DN)

In [ ]:
from sklearn.metrics import (average_precision_score, log_loss, confusion_matrix,
                            precision_recall_fscore_support, f1_score)

import matplotlib.pyplot as plt


import os

from ruamel.yaml import YAML
import pandas as pd
import numpy as np
import joblib

from collections import defaultdict
from itertools import chain

import click
import json

import torch
import torch.nn as nn
from torch.optim.lr_scheduler import ExponentialLR, MultiStepLR
from torch.utils.data import DataLoader, Dataset


from transformers import BertTokenizer, BertForSequenceClassification
from transformers import AutoTokenizer, AutoModelForMaskedLM, BertConfig, AutoModel
from transformers import DataCollatorWithPadding
from transformers import RobertaTokenizer, RobertaModel

DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

import sys
sys.path.append('.')
from src.funcs import set_seed
from src.funcs import metric_multi
from src.funcs import get_opt_thresh, get_preds
from src.funcs import get_conf_df
from src.spec_nn_funcs import TextDFDataset, TextModelClass
from ruamel.yaml import YAML

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import RobustScaler

from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier

In [ ]:


conf = YAML().load(open('params.yaml'))
conf_ttp = YAML().load(open('dvc_pipes/ttp/params_ttp.yaml'))
conf_bert = YAML().load(open('dvc_pipes/bert/params_bert.yaml'))
conf_bert_ttp = YAML().load(open('dvc_pipes/bert_ttp/params_bert_ttp.yaml'))

set_seed(conf['seed'])

In [ ]:
bert_type = conf_bert['nn_bert']['bert_type']


In [ ]:
VALID_BATCH_SIZE = conf_bert['nn']['batch_size']
TRAIN_BATCH_SIZE = conf_bert['nn']['batch_size']
MAX_SEQ_LENGTH = conf_bert['nn']['maxlen']

checkpoint = 'data/external/models/SecureBERT_Plus/snapshots/4c48ccdb8d2019f179b07dfa27656c655394d78e'
tokenizer = RobertaTokenizer.from_pretrained(checkpoint)
tokenizer_opts = {'max_length':MAX_SEQ_LENGTH, 'return_tensors':"pt", 'padding':True, 'truncation':True, 'add_special_tokens':True}


# Предсказания техник

In [ ]:
mlb_ttp = joblib.load(conf['prep_text']['ttp_mlb_fn'])
mlb_sub = joblib.load('data/temp/subt/ttp/mlb.pkl')

In [ ]:
data_ttp = pd.read_csv(conf_ttp['feat_gen_ttp']['data_fn'])
data_ttp['target'] = data_ttp['target'].map(lambda x: eval(x))
data_ttp['ttp'] = data_ttp['ttp'].map(lambda x: eval(x))

data_sub = pd.read_csv('data/temp/subt/ttp/data_df.csv')
data_sub['target'] = data_sub['target'].map(lambda x: eval(x))
data_sub['ttp'] = data_sub['ttp'].map(lambda x: eval(x))

In [ ]:
model_bert_ttp = torch.load(f'{conf_bert_ttp["nn_bert_ttp"]["model_fn"]}')

model_bert_sub = torch.load(f'data/temp/subt/model.pt')

In [ ]:
feat_ttp = pd.read_csv(conf_ttp['feat_eng_ttp']['feat_final_fn'])
feat_sub = pd.read_csv('data/temp/subt/ttp/feat_final_df.csv')



In [ ]:
feat_sub.shape, feat_ttp.shape

In [ ]:
tr_ttp_ds = TextDFDataset(data_ttp.query('split=="tr"').reset_index(drop=True), tokenizer=tokenizer, tokenizer_opts=tokenizer_opts)
val_ttp_ds = TextDFDataset(data_ttp.query('split=="val"').reset_index(drop=True), tokenizer=tokenizer, tokenizer_opts=tokenizer_opts)
ts_ttp_ds = TextDFDataset(data_ttp.query('split=="ts"').reset_index(drop=True), tokenizer=tokenizer, tokenizer_opts=tokenizer_opts)

tr_ttp_ld = DataLoader(tr_ttp_ds, batch_size = TRAIN_BATCH_SIZE, shuffle = False, collate_fn = DataCollatorWithPadding(tokenizer=tokenizer))
val_ttp_ld = DataLoader(val_ttp_ds, batch_size = VALID_BATCH_SIZE, shuffle = False, collate_fn = DataCollatorWithPadding(tokenizer=tokenizer))
ts_ttp_ld = DataLoader(ts_ttp_ds, batch_size = VALID_BATCH_SIZE, shuffle = False, collate_fn = DataCollatorWithPadding(tokenizer=tokenizer))


In [ ]:
tr_sub_ds = TextDFDataset(data_sub.query('split=="tr"').reset_index(drop=True), tokenizer=tokenizer, tokenizer_opts=tokenizer_opts)
val_sub_ds = TextDFDataset(data_sub.query('split=="val"').reset_index(drop=True), tokenizer=tokenizer, tokenizer_opts=tokenizer_opts)
ts_sub_ds = TextDFDataset(data_sub.query('split=="ts"').reset_index(drop=True), tokenizer=tokenizer, tokenizer_opts=tokenizer_opts)

tr_sub_ld = DataLoader(tr_sub_ds, batch_size = TRAIN_BATCH_SIZE, shuffle = False, collate_fn = DataCollatorWithPadding(tokenizer=tokenizer))
val_sub_ld = DataLoader(val_sub_ds, batch_size = VALID_BATCH_SIZE, shuffle = False, collate_fn = DataCollatorWithPadding(tokenizer=tokenizer))
ts_sub_ld = DataLoader(ts_sub_ds, batch_size = VALID_BATCH_SIZE, shuffle = False, collate_fn = DataCollatorWithPadding(tokenizer=tokenizer))


In [ ]:
Y_ttp_val_proba = np.array(get_preds(model_bert_ttp, ld=val_ttp_ld)['pred'])
Y_ttp_tr_proba = np.array(get_preds(model_bert_ttp, ld=tr_ttp_ld)['pred'])
Y_ttp_ts_proba = np.array(get_preds(model_bert_ttp, ld=ts_ttp_ld)['pred'])

In [ ]:
Y_sub_val_proba = np.array(get_preds(model_bert_sub, ld=val_sub_ld)['pred'])
Y_sub_tr_proba = np.array(get_preds(model_bert_sub, ld=tr_sub_ld)['pred'])
Y_sub_ts_proba = np.array(get_preds(model_bert_sub, ld=ts_sub_ld)['pred'])

In [ ]:
thresh_ttp_l = get_opt_thresh(y_true = np.array(data_ttp.loc[data_ttp.split=='tr', 'target'].values.tolist()), 
                          probas = Y_ttp_tr_proba, mlb = mlb_ttp, 
                          opt_metric=conf['train_eval_model']['opt_metric'], 
                          thresh_space_l=np.arange(0.001, 1, 0.002), dump_fn = None)

thresh_sub_l = get_opt_thresh(y_true = np.array(data_sub.loc[data_sub.split=='tr', 'target'].values.tolist()), 
                          probas = Y_sub_tr_proba, mlb = mlb_sub, 
                          opt_metric=conf['train_eval_model']['opt_metric'], 
                          thresh_space_l=np.arange(0.001, 1, 0.002), dump_fn = None)



In [ ]:
ttp_df = pd.concat([data_ttp[['sentence', 'ttp', 'labels',	'url', 'target', 'split']].query('split=="tr"').assign(proba_ttp = Y_ttp_tr_proba.tolist()),
          data_ttp[['sentence', 'ttp', 'labels',	'url', 'target', 'split']].query('split=="val"').assign(proba_ttp = Y_ttp_val_proba.tolist()),
           data_ttp[['sentence', 'ttp', 'labels',	'url', 'target', 'split']].query('split=="ts"').assign(proba_ttp = Y_ttp_ts_proba.tolist())
          ], axis=0, ignore_index=True)

ttp_df['pred_ttp'] = ttp_df['proba_ttp'].map(lambda x: [int(val>=thresh) for val, thresh in zip(x, thresh_ttp_l)])


sub_df = pd.concat([data_sub[['sentence', 'ttp', 'labels',	'url', 'target', 'split']].query('split=="tr"').assign(proba_ttp = Y_sub_tr_proba.tolist()),
          data_sub[['sentence', 'ttp', 'labels',	'url', 'target', 'split']].query('split=="val"').assign(proba_ttp = Y_sub_val_proba.tolist()),
           data_sub[['sentence', 'ttp', 'labels',	'url', 'target', 'split']].query('split=="ts"').assign(proba_ttp = Y_sub_ts_proba.tolist())
          ], axis=0, ignore_index=True)

sub_df['pred_ttp'] = sub_df['proba_ttp'].map(lambda x: [int(val>=thresh) for val, thresh in zip(x, thresh_sub_l)])

In [ ]:
ttp_df['pred_str_ttp'] = ttp_df['pred_ttp'].map(lambda x: mlb_ttp.inverse_transform(np.array([x]))[0])
sub_df['pred_str_ttp'] = sub_df['pred_ttp'].map(lambda x: mlb_sub.inverse_transform(np.array([x]))[0])

# Сверка

## pr_auc

In [ ]:
pr_ts, pr_ts_l = metric_multi(np.array(ttp_df.query('split=="ts"')['target'].values.tolist()), 
                            np.array(ttp_df.query('split=="ts"')['proba_ttp'].values.tolist()),
                            average_precision_score)

pr_val, pr_val_l = metric_multi(np.array(ttp_df.query('split=="val"')['target'].values.tolist()), 
                            np.array(ttp_df.query('split=="val"')['proba_ttp'].values.tolist()),
                            average_precision_score)

pr_tr, pr_tr_l = metric_multi(np.array(ttp_df.query('split=="tr"')['target'].values.tolist()), 
                            np.array(ttp_df.query('split=="tr"')['proba_ttp'].values.tolist()),
                            average_precision_score)


pr_tr, pr_val, pr_ts

In [ ]:
pr_ts, pr_ts_l = metric_multi(np.array(sub_df.query('split=="ts"')['target'].values.tolist()), 
                            np.array(sub_df.query('split=="ts"')['proba_ttp'].values.tolist()),
                            average_precision_score)

pr_val, pr_val_l = metric_multi(np.array(sub_df.query('split=="val"')['target'].values.tolist()), 
                            np.array(sub_df.query('split=="val"')['proba_ttp'].values.tolist()),
                            average_precision_score)


pr_tr, pr_tr_l = metric_multi(np.array(sub_df.query('split=="tr"')['target'].values.tolist()), 
                            np.array(sub_df.query('split=="tr"')['proba_ttp'].values.tolist()),
                            average_precision_score)



pr_tr, pr_val, pr_ts

## f1

In [ ]:
p_val_micro, r_val_micro, f1_val_micro, sup = precision_recall_fscore_support(np.array(ttp_df.query('split=="val"')['target'].values.tolist()), 
                                                    np.array(ttp_df.query('split=="val"')['pred_ttp'].values.tolist()), average='micro')

p_val_macro, r_val_macro, f1_val_macro, sup = precision_recall_fscore_support(np.array(ttp_df.query('split=="val"')['target'].values.tolist()), 
                                                    np.array(ttp_df.query('split=="val"')['pred_ttp'].values.tolist()), average='macro')

f1_val_micro, f1_val_macro

In [ ]:
p_val_micro, r_val_micro, f1_val_micro, sup = precision_recall_fscore_support(np.array(sub_df.query('split=="val"')['target'].values.tolist()), 
                                                    np.array(sub_df.query('split=="val"')['pred_ttp'].values.tolist()), average='micro')
p_val_macro, r_val_macro, f1_val_macro, sup = precision_recall_fscore_support(np.array(sub_df.query('split=="val"')['target'].values.tolist()), 
                                                    np.array(sub_df.query('split=="val"')['pred_ttp'].values.tolist()), average='macro')

f1_val_micro, f1_val_macro

### для субтехник не считаем ошибкой все, что до точки

In [ ]:
sub_df = pd.concat([data_sub[['sentence', 'ttp', 'labels',	'url', 'target', 'split']].query('split=="tr"').assign(proba_ttp = Y_sub_tr_proba.tolist()),
          data_sub[['sentence', 'ttp', 'labels',	'url', 'target', 'split']].query('split=="val"').assign(proba_ttp = Y_sub_val_proba.tolist()),
           data_sub[['sentence', 'ttp', 'labels',	'url', 'target', 'split']].query('split=="ts"').assign(proba_ttp = Y_sub_ts_proba.tolist())
          ], axis=0, ignore_index=True)

sub_df['pred_ttp'] = sub_df['proba_ttp'].map(lambda x: [int(val>=thresh) for val, thresh in zip(x, thresh_sub_l)])
sub_df['pred_str_ttp'] = sub_df['pred_ttp'].map(lambda x: mlb_sub.inverse_transform(np.array([x]))[0])

In [ ]:
Y_val_proba = Y_sub_val_proba
thresh_l = thresh_sub_l
Y_val = np.array(data_sub.loc[data_sub.split=='val', 'target'].values.tolist())

In [ ]:
res_df = pd.DataFrame()
res_df['y'] = Y_val.tolist()
res_df['y_proba'] = Y_val_proba.tolist()

thresh_col = 'y_p'

res_df[thresh_col] = res_df['y_proba'].map(lambda x: [int(val>=thresh) for val, thresh in zip(x, thresh_l)])


error_df = data_sub.query('split=="val"').reset_index(names='val_idx').join(res_df[['y', 'y_proba', thresh_col]])
error_df['prob_label'] = mlb_sub.inverse_transform(np.array(error_df[thresh_col].tolist()))

p_val_micro, r_val_micro, f1_val_micro, sup = precision_recall_fscore_support(np.array(res_df['y'].values.tolist()), 
                                                    np.array(res_df[thresh_col].values.tolist()), average='micro')
p_val_macro, r_val_macro, f1_val_macro, sup = precision_recall_fscore_support(np.array(res_df['y'].values.tolist()), 
                                                    np.array(res_df[thresh_col].values.tolist()), average='macro')


f1_val_micro, f1_val_macro

In [ ]:
t_l = [[it.split('.')[0]] for it in data_sub.explode('ttp')['ttp'].dropna().unique()]

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb_f1 = MultiLabelBinarizer()
mlb_f1.fit(t_l)


In [ ]:
error_df['pred_enc'] = error_df['prob_label'].map(lambda x: [it.split('.')[0] for it in x]).tolist()
error_df['pred_enc'] = mlb_f1.transform(error_df['pred_enc']).tolist()

error_df['ttp_enc'] = error_df['ttp'].map(lambda x: [it.split('.')[0] for it in x]).tolist()
error_df['ttp_enc'] = mlb_f1.transform(error_df['ttp_enc']).tolist()

In [ ]:
p_val_micro, r_val_micro, f1_val_micro, sup = precision_recall_fscore_support(np.array(error_df['ttp_enc'].values.tolist()), 
                                                    np.array(error_df['pred_enc'].values.tolist()), average='micro')
p_val_macro, r_val_macro, f1_val_macro, sup = precision_recall_fscore_support(np.array(error_df['ttp_enc'].values.tolist()), 
                                                    np.array(error_df['pred_enc'].values.tolist()), average='macro')

f1_val_micro, f1_val_macro

In [ ]:
_, res_l = metric_multi(np.array(error_df['ttp_enc'].tolist()), np.array(error_df['pred_enc'].tolist()), f1_score)

pd.DataFrame({'qual':res_l, 'class':mlb_f1.classes_}).sort_values(by='qual').head(20)


# Анализ

In [ ]:
data_ttp.explode('ttp').groupby(['split', 'ttp']).size().unstack()

## Выборки одинаковые

In [ ]:
data_ttp.groupby('split').size()

In [ ]:
data_sub.groupby('split').size()

In [ ]:
ttp_df.loc[(ttp_df.split!="tr") , ['sentence', 'ttp', 'labels', 'pred_str_ttp']].merge(
    sub_df.loc[(sub_df.split!="tr"), ['sentence', 'ttp', 'labels', 'pred_str_ttp']], on='sentence', how='inner'
)

# Bert by class сравнить

In [ ]:
by_ttp_df = pd.read_csv('data/out/bert_ttp/bert_by_class_metric.csv')
by_sub_df = pd.read_csv('data/temp/subt/bert_by_class_metric.csv')


In [ ]:
diff_classes = by_sub_df.merge(by_ttp_df, on='class').assign(diff=lambda x: abs(x['qual_x']-x['qual_y'])).sort_values(by='diff', ascending=False)['class'].head(10)
by_sub_df.merge(by_ttp_df, on='class').assign(diff=lambda x: abs(x['qual_x']-x['qual_y'])).sort_values(by='diff', ascending=False).head(10)

# main_diff

In [ ]:
N = 1
ttp = diff_classes.iloc[N]
ttp

In [ ]:
by_sub_df[by_sub_df['class'].map(lambda x: ttp in x)]

In [ ]:
data_ttp.explode('ttp').groupby(['split', 'ttp']).size().unstack()[ttp]

In [ ]:
data_sub.explode('ttp').groupby(['split', 'ttp']).size().unstack()[ttp]

In [ ]:
ttp_df.loc[(ttp_df.split=="val") & (ttp_df.ttp.map(lambda x: ttp in x)), ['sentence', 'ttp', 'labels', 'pred_str_ttp']].merge(
    sub_df.loc[(sub_df.split=="val") & (sub_df.ttp.map(lambda x: ttp in x)), ['sentence', 'ttp', 'labels', 'pred_str_ttp']], on='sentence', how='outer'
)

In [ ]:
data = pd.read_csv(conf['prep_text']['prep_fn'])


data['ttp'] = data['ttp'].map(lambda x: eval(x))
synth_thresh_class_num = 20


sel_dop = (data.split=='tr')
mini_ttp_l = data[sel_dop].explode('ttp').groupby('ttp').size().loc[lambda x: x<=synth_thresh_class_num].index.tolist()

ttp in mini_ttp_l

In [ ]:
data_s= pd.read_csv('data/temp/subt/ttp/prep_df.csv')

data_s['ttp'] = data_s['ttp'].map(lambda x: eval(x))
synth_thresh_class_num = 20

sel_dop = (data_s.split=='tr')
mini_ttp_l = data_s[sel_dop].explode('ttp').groupby('ttp').size().loc[lambda x: x<=synth_thresh_class_num].index.tolist()

ttp in mini_ttp_l

## найдем thresh

In [ ]:
ttp_idx = np.where(mlb_ttp.classes_==ttp)[0][0]
sub_idx = np.where(mlb_sub.classes_==ttp)[0][0]

In [ ]:
thresh_ttp_l[ttp_idx], thresh_sub_l[sub_idx]

<div class='alert alert-info'> 
    
    - T1071 - с подтехниками, сам класс плохо определяется на уровне субтехник, но его подтехники нормально
    - T1614 - с подтехниками чистый класс аугментировался, и хуже предсказался, хотя в одном случае - T1614.001 (но формально это ошибка). Хотя все подклассы объединены в метку T1614, поэтому даже, где техника кажется правильной, может быть подтехника (это те, где nan в пересечении ttp_df и sub_df)
    - T1134 как и T1614, хотя f1  ноль для чистого, в 2 случаях субкласс предсказался, хотя формально и ошибка, алгоритм почуял верно. T1134 интересно, что теряем подтехники маленькие (в rare), хотя могли их отнести к классу
    - T1036 аналогично
    - T1059 - если считать не только чистый, но и подтехники, то качество норма
    - T1071 - f1 за счет подтехник, которые да, лучше, чем на уровне подтехник предсказываются
    - T1221, T1040 - аугментация в пользу субтехник
    - T1554 - на один пример лучше предсказано для техник, аугментация поточнее или другие факторы
</div>

<div class='alert alert-info'> TO DO
    
    - T1134 - теряем подтехники маленькие (в rare), хотя могли их отнести к классу. Может так  сделать?
    - может вывести для субтехник метрику f1 для классов в предположении, что target до точки и предсказание, чтобы не считать ошибки на уровне субтехник
</div>

# T1529

In [ ]:
ttp = 'T1529'

In [ ]:
data_ttp.explode('ttp').groupby(['split', 'ttp']).size().unstack()[ttp]

In [ ]:
data_sub.explode('ttp').groupby(['split', 'ttp']).size().unstack()[ttp]

In [ ]:
ttp_df.loc[(ttp_df.split=="val") & (ttp_df.ttp.map(lambda x: ttp in x)), ['sentence', 'ttp', 'labels', 'pred_str_ttp']].merge(
    sub_df.loc[(sub_df.split=="val") & (sub_df.ttp.map(lambda x: ttp in x)), ['sentence', 'ttp', 'labels', 'pred_str_ttp']], on='sentence', how='outer'
)

In [ ]:
data = pd.read_csv(conf['prep_text']['prep_fn'])


data['ttp'] = data['ttp'].map(lambda x: eval(x))
synth_thresh_class_num = 20


sel_dop = (data.split=='tr')
mini_ttp_l = data[sel_dop].explode('ttp').groupby('ttp').size().loc[lambda x: x<=synth_thresh_class_num].index.tolist()

ttp in mini_ttp_l

In [ ]:
data_s= pd.read_csv('data/temp/subt/ttp/prep_df.csv')

data_s['ttp'] = data_s['ttp'].map(lambda x: eval(x))
synth_thresh_class_num = 20

sel_dop = (data_s.split=='tr')
mini_ttp_l = data_s[sel_dop].explode('ttp').groupby('ttp').size().loc[lambda x: x<=synth_thresh_class_num].index.tolist()

ttp in mini_ttp_l

In [ ]:
data_s[sel_dop].explode('ttp').groupby('ttp').size().loc[lambda x: x<=synth_thresh_class_num][ttp]

То есть 13 кейсов только

## найдем thresh

In [ ]:
ttp_idx = np.where(mlb_ttp.classes_==ttp)[0][0]
sub_idx = np.where(mlb_sub.classes_==ttp)[0][0]

In [ ]:
thresh_ttp_l[ttp_idx], thresh_sub_l[sub_idx]

<div class='alert alert-info'>
и train выборки и границы сильно отличаются
</div>

In [ ]:
conf_bert_ttp['nn_bert_ttp']['opt_metric_fn']

In [ ]:
opt_metric_sub_df = pd.read_csv('data/temp/subt/ttp/bert_opt_metric.csv').set_index('Unnamed: 0').loc[lambda x:x['class_nm']==ttp]
idx_sub = opt_metric_sub_df['f1'].argmax()

In [ ]:
opt_metric_df = pd.read_csv(conf_bert_ttp['nn_bert_ttp']['opt_metric_fn']).set_index('Unnamed: 0').loc[lambda x:x['class_nm']==ttp]

idx = opt_metric_df['f1'].argmax()

In [ ]:
opt_metric_df.iloc[idx-2:idx+2]


In [ ]:
opt_metric_sub_df.iloc[60:70]

In [ ]:
# FP, видимо, были до 
opt_metric_sub_df.iloc[idx_sub-2:idx_sub+2]


In [ ]:
comp_df = ttp_df.loc[(ttp_df.ttp.map(lambda x: ttp in x))&(ttp_df.split=="tr"), ['sentence','proba_ttp', 'pred_str_ttp']]\
        .assign(proba_ttp=lambda x:x['proba_ttp'].map(lambda y:y[ttp_idx]))\
    .merge(
sub_df.loc[(sub_df.ttp.map(lambda x: ttp in x))&(sub_df.split=="tr"), ['sentence','proba_ttp', 'pred_str_ttp']]\
        .assign(proba_ttp=lambda x:x['proba_ttp'].map(lambda y:y[sub_idx])), on='sentence', how='outer'
        
    )

In [ ]:
comp_df.dropna()

# T1124

In [ ]:
ttp = 'T1124'

In [ ]:
data_ttp.explode('ttp').groupby(['split', 'ttp']).size().unstack()[ttp]

In [ ]:
data_sub.explode('ttp').groupby(['split', 'ttp']).size().unstack()[ttp]

In [ ]:
ttp_df.loc[(ttp_df.split=="val") & (ttp_df.ttp.map(lambda x: ttp in x)), ['sentence', 'ttp', 'labels', 'pred_str_ttp']].merge(
    sub_df.loc[(sub_df.split=="val") & (sub_df.ttp.map(lambda x: ttp in x)), ['sentence', 'ttp', 'labels', 'pred_str_ttp']], on='sentence', how='outer'
)

# T1010

In [ ]:
ttp = 'T1010'

In [ ]:
data_ttp.explode('ttp').groupby(['split', 'ttp']).size().unstack()[ttp]

In [ ]:
data_sub.explode('ttp').groupby(['split', 'ttp']).size().unstack()[ttp]

In [ ]:
by_sub_df.merge(by_ttp_df, on='class').assign(diff=lambda x: abs(x['qual_x']-x['qual_y'])).sort_values(by='diff', ascending=False).loc[lambda x: x['class']=='T1010']

<div class='alert alert-info'> Качество сопоставимое
</div>

# T1014

In [ ]:
ttp = 'T1014'

In [ ]:
data_ttp.explode('ttp').groupby(['split', 'ttp']).size().unstack()[ttp]

In [ ]:
data_sub.explode('ttp').groupby(['split', 'ttp']).size().unstack()[ttp]

In [ ]:
ttp_df.loc[(ttp_df.split=="val") & (ttp_df.ttp.map(lambda x: ttp in x)), ['sentence', 'ttp', 'labels', 'pred_str_ttp']].merge(
    sub_df.loc[(sub_df.split=="val") & (sub_df.ttp.map(lambda x: ttp in x)), ['sentence', 'ttp', 'labels', 'pred_str_ttp']], on='sentence', how='outer'
)

- тут разные train выборки для этого класса, так как без субтехник класс попал в 20-ку тех, где аугментация применяется

In [ ]:
data = pd.read_csv(conf['prep_text']['prep_fn'])


data['ttp'] = data['ttp'].map(lambda x: eval(x))
synth_thresh_class_num = 20


sel_dop = (data.split=='tr')
mini_ttp_l = data[sel_dop].explode('ttp').groupby('ttp').size().loc[lambda x: x<=synth_thresh_class_num].index.tolist()

ttp in mini_ttp_l

# Причины

Причины расхождений:
-  Основная. Разные выборки - ВСЕ, так как шла стратификация (выравнивание выборок) по разным целям (для субтехник выравнивание по субтехникам, а после их обобщения до техник, шло выравнивание выборок по техникам). В какую-то попали более удачные записи, и пара записей сильно меняла процентное соотношение, а качество предсказаний для всех записей примерно одинаковое, что подтверждается при унификации выборок. Обновленные файлы высылаю.
- Для чистых классов, где-то 0 в f1, хотя подтехники не плохо определяются (например, T1071, T1059). Чтобы повысить справедливость сравнения, убрал ошибку, связанную с угадыванием подтехники, но неугадыванием самой техники (так как формально для субтехник это ошибка, а для модели по техникам - нет).
- 20-ка самых малых классов для аугментации тоже по-разному определена, где-то из-за этого разный cut-off на train, соответственно, разные результаты на валидации.
- Общие слои в берте по-разному обучаются, так как таргет разный, выборки и пакеты 

После унификации выборок:
- f1_val_micro, f1_val_macro для техник - (0.6067165366862963, 0.5265788800947466)
- f1_val_micro, f1_val_macro для субтехник - (0.5522865366165653, 0.4274561424537244). После того, как убрал ошибку при неугадывании техники, но угадывании подтехники - (0.5866231647634584, 0.4889006130861287)

In [ ]:
_, res_l = metric_multi(np.array(error_df['ttp_enc'].tolist()), np.array(error_df['pred_enc'].tolist()), f1_score)

pd.DataFrame({'qual':res_l, 'class':mlb_f1.classes_}).sort_values(by='qual').loc[lambda x: x['class'].isin(["T1059", "T1071", "T1014"])]


In [ ]:
pd.DataFrame({'qual':res_l, 'class':mlb_f1.classes_}).sort_values(by='qual').loc[lambda x: x['class'].isin(["T1040", "T1072", "T1014", "T1037", "T1221"])]